#Please note - I have changed and tested with various parameters. Also need to update to all the directories and paths for data where needed.

In [ ]:
import zipfile
import os

ZIP_PATH = ""
EXTRACT_PATH = ""

if not os.path.exists(ZIP_PATH):
    print(f"zip file not found {ZIP_PATH}")
else:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        # Check what is actually inside the ZIP
        file_list = zip_ref.namelist()
        zip_ref.extractall(EXTRACT_PATH)

    all_files = []
    for root, dirs, files in os.walk(EXTRACT_PATH):
        for file in files:
            all_files.append(os.path.join(root, file))
    

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification, TrainingArguments, Trainer
from torchvision.transforms import Compose, Resize, ToTensor, Normalize

CSV_FILE = ""
BASE_FRAME_PATH = ""


MODEL_CKPT = "MCG-NJU/videomae-small-finetuned-kinetics"

label_to_id = {"High": 0, "Low": 1, "Medium": 2}
id_label = {v: k for k, v in label_to_id.items()}

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=3.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma * ce_loss)

        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss.sum()

class SlidingWindowDataset(Dataset):
    def __init__(self, dataframe, base_path, window_size=16, stride=8, transform=None):
        self.base_path = base_path
        self.transform = transform
        self.window_size = window_size
        self.stride = stride
        self.samples = []

        for _, row in dataframe.iterrows():
            video_name = str(row['video_filename'])
            label = label_to_id[row['engagement_label']]
            video_folder = os.path.join(self.base_path, video_name)

            if not os.path.exists(video_folder):
                continue

            frame_files = sorted([f for f in os.listdir(video_folder) if f.endswith(('.jpg', '.png'))])
            num_total_frames = len(frame_files)

            for start in range(0, num_total_frames - window_size + 1, stride):
                end = start + window_size
                window_frames = frame_files[start:end]
                self.samples.append({
                    "video_folder": video_folder,
                    "frame_paths": window_frames,
                    "label": label
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        frames = []
        for f_name in sample["frame_paths"]:
            img_path = os.path.join(sample["video_folder"], f_name)
            img = Image.open(img_path).convert("RGB")
            if self.transform:
                img = self.transform(img)
            frames.append(img)

        return {
            "pixel_values": torch.stack(frames),
            "labels": torch.tensor(sample["label"])
        }

df = pd.read_csv(CSV_FILE)
df.columns = df.columns.str.strip()

processor = VideoMAEImageProcessor.from_pretrained(MODEL_CKPT)
video_transform = Compose([
    Resize((224, 224)),
    ToTensor(),
    Normalize(mean=processor.image_mean, std=processor.image_std),
])


train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['engagement_label'])

train_dataset = SlidingWindowDataset(train_df, BASE_FRAME_PATH, window_size=16, stride=8, transform=video_transform)
val_dataset = SlidingWindowDataset(val_df, BASE_FRAME_PATH, window_size=16, stride=8, transform=video_transform)


manual_weights = torch.tensor([1.2, 2.0, 0.2], dtype=torch.float).to("cuda")

class VideoTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = FocalLoss(alpha=manual_weights, gamma=3.0)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

model = VideoMAEForVideoClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=3,
    label2id=label_to_id,
    id2label=id_label,
    ignore_mismatched_sizes=True
)


training_args = TrainingArguments(
    output_dir="./videomae_exp",
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,          
    weight_decay=0.05,           
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=12,         
    fp16=True,
    load_best_model_at_end=True,
    logging_steps=10,
)

trainer = VideoTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

In [ ]:
output = trainer.predict(val_dataset)
y_pred = np.argmax(output.predictions, axis=-1)
y_true = output.label_ids

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print(f"Best Accuracy: {accuracy_score(y_true, y_pred) * 100:.2f}%")
print(classification_report(y_true, y_pred, target_names=["High", "Low", "Medium"]))